In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

# Define hyperparameters (same as used during training)
image_size = 64

# Define transformations for preprocessing the input image
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Define the critic (discriminator) network
class Discriminator(nn.Module):

    def __init__(self, channels_img, features_d):
        super(Discriminator, self).__init__()

        self.disc = nn.Sequential(
            #size = 3*64*64
            nn.Conv2d(channels_img, features_d, kernel_size = 4, stride = 2, padding = 1), # Size : 32*32
            nn.LeakyReLU(0.2),

            nn.Conv2d(features_d, features_d*2, kernel_size = 4, stride = 2, padding = 1), # size = 16*16
            nn.BatchNorm2d(features_d*2),
            nn.LeakyReLU(0.2),

            nn.Conv2d(features_d*2, features_d*4, kernel_size = 4, stride = 2, padding = 1), # size = 8*8
            nn.BatchNorm2d(features_d*4) ki,
            nn.LeakyReLU(0.2),

            nn.Conv2d(features_d*4, features_d*8, kernel_size = 4, stride = 2, padding = 1), # size = 4*4
            nn.BatchNorm2d(features_d*8),
            nn.LeakyReLU(0.2),

            nn.Conv2d(features_d*8, 1, kernel_size = 4, stride = 2, padding = 0) #1*1

        )


    def forward(self, x):
        return self.disc(x)

# Load the trained discriminator model
discriminator = Discriminator(channels_img=3, features_d=64)  # You need to provide the appropriate features_d value
state_dict = torch.load("/content/drive/MyDrive/W_training/discriminator_epoch350.pth")
discriminator.load_state_dict(state_dict, strict=False)
discriminator.eval()  # Set the discriminator to evaluation mode


#discriminator.load_state_dict(state_dict)
#discriminator.eval()  # Set the discriminator to evaluation mode



# Function to classify the image as fake or genuine
def classify_image(image_path, threshold=0.5):
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0)

    # Use the discriminator to classify the image
    with torch.no_grad():
        output = discriminator(image)

    # Rescale the discriminator's output to the range [0, 1]
    output = torch.sigmoid(output).item()

    # Debug: Print the discriminator's output
    print(f"Discriminator output: {output}")

    # Check if the image is fake or genuine based on the threshold
    if output > threshold:
        return "1"
    else:
        return "0"

# Test an input image (replace the image path with the path to the forged/generated image)
#image_path_to_test = "/content/drive/MyDrive/TEST/Copy of O_0003.jpg"
#result = classify_image(image_path_to_test)

#print(f"The image is classified as: {result}")
# Test an input image for different thresholds
image_path_to_test = "/content/drive/MyDrive/TEST/cm_00159.jpg"  # Replace with the path of the image you want to test
thresholds = [0.2 + i * 0.01 for i in range(0, 40)]

for threshold in thresholds:
    result = classify_image(image_path_to_test, threshold=threshold)
    print(f"The image is classified as: {result} (Threshold: {threshold})")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/W_training/discriminator_epoch350.pth'

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

# Define hyperparameters (same as used during training)
image_size = 64
image_channels = 3

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define transformations for preprocessing the input image
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Define the Discriminator class
class Discriminator(nn.Module):
    def __init__(self, image_channels, hidden_dim=64):
        super(Discriminator, self).__init__()

        self.main = nn.Sequential(
            # Input: image_channels x 64 x 64
            nn.Conv2d(image_channels, hidden_dim, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # Input: hidden_dim x 32 x 32
            nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(hidden_dim * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # Input: hidden_dim * 2 x 16 x 16
            nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(hidden_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # Input: hidden_dim * 4 x 8 x 8
            nn.Conv2d(hidden_dim * 4, hidden_dim * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(hidden_dim * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # Input: hidden_dim * 8 x 4 x 4
            nn.Conv2d(hidden_dim * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()  # Output: 1 x 1 x 1
        )

    def forward(self, input):
      return self.main(input)

# Load the pre-trained discriminator model on the CPU
discriminator = Discriminator(image_channels).to(device)
discriminator.load_state_dict(torch.load("/content/drive/MyDrive/dcgan/netD_epoch_450.pth", map_location=device))
discriminator.eval()  # Set the discriminator to evaluation mode

# Define the classify_image function
def classify_image(image_path, threshold=0.8):
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    # Use the discriminator to classify the image
    with torch.no_grad():
        output = discriminator(image)

    # Apply a sigmoid activation function to get the probabilities
    probabilities = torch.sigmoid(output)

    # Compute the mean probability over the batch
    mean_probability = torch.mean(probabilities).item()

    # Check if the image is fake or genuine based on the mean probability and threshold
    if mean_probability > threshold:
        return "1"
    else:
        return "0"



import os
import multiprocessing

# Directory containing the image files
image_dir = "/content/drive/MyDrive/GATE/Original_images"  # Replace with the path to your image directory

# Threshold for classification
threshold = 0.5

# List all image files in the directory
image_files = [os.path.join(image_dir, filename) for filename in os.listdir(image_dir) if filename.endswith((".jpg", ".png", ".jpeg"))]

# Define a function for parallel processing
def classify_and_print_result(image_path):
    result = classify_image(image_path, threshold=threshold)
    print(f"The image at {image_path} is classified as: {result}")

if __name__ == "__main__":
    # Create a multiprocessing pool with the number of desired processes
    num_processes = multiprocessing.cpu_count()  # You can adjust this as needed
    pool = multiprocessing.Pool(processes=num_processes)

    # Use the pool to classify images in parallel
    pool.map(classify_and_print_result, image_files)

    # Close the pool to release resources
    pool.close()
    pool.join()


# Test an input image for different thresholds
#image_path_to_test = "/content/drive/MyDrive/TEST/Copy of O_0001.jpg"  # Replace with the path of the image you want to test
#thresholds = [0.5 + i * 0.01 for i in range(0, 20)]

#for threshold in thresholds:
    #result = classify_image(image_path_to_test, threshold=threshold)
    #print(f"The image is classified as: {result} (Threshold: {threshold})")


The image at /content/drive/MyDrive/GATE/Original_images/O_0023.jpg is classified as: 1
The image at /content/drive/MyDrive/GATE/Original_images/O_0021.jpg is classified as: 1
The image at /content/drive/MyDrive/GATE/Original_images/O_0026.jpg is classified as: 1
The image at /content/drive/MyDrive/GATE/Original_images/O_0014.jpg is classified as: 1
The image at /content/drive/MyDrive/GATE/Original_images/O_0010.jpg is classified as: 1
The image at /content/drive/MyDrive/GATE/Original_images/O_0016.jpg is classified as: 1
The image at /content/drive/MyDrive/GATE/Original_images/O_0024.jpg is classified as: 1The image at /content/drive/MyDrive/GATE/Original_images/O_0019.jpg is classified as: 1

The image at /content/drive/MyDrive/GATE/Original_images/O_0015.jpg is classified as: 1
The image at /content/drive/MyDrive/GATE/Original_images/O_0013.jpg is classified as: 1
The image at /content/drive/MyDrive/GATE/Original_images/O_0025.jpg is classified as: 1The image at /content/drive/MyDri